In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
# OBTENEMOS LOS DATOS DE LA CAPA SILVER
gold_silver = spark.table("spotify_catalog.silver.spotify_tracks")

display(gold_silver.limit(10))

### CREAMOS LA DIMENSIÓN ARTIST

In [0]:
dim_artist_base = (
    gold_silver
    .select(
        "artist_id",
        "artist_name",
        "artist_href",
        "artist_uri"
    )
    .filter(col("artist_id").isNotNull())
    .dropDuplicates(["artist_id"])
)

In [0]:
dim_artist = (
    dim_artist_base
    .withColumn(
        "sk_artist",
        row_number().over(window_artist)
    )
    .select(
        "sk_artist",
        "artist_id",
        "artist_name",
        "artist_href",
        "artist_uri"
    )
)

display(dim_artist)

In [0]:
(
    dim_artist
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "spotify_catalog.gold.dim_artist"
    )
)